# 🎯 LLM Instruction Fine-Tuning: A Learning Journey

> **Philosophy of this notebook:**  
> We intentionally show the *wrong* approach first, hit the real error, understand *why* it fails,  
> then evolve the code to the correct solution. Each section builds on the previous.

---

## Where does Instruction Fine-Tuning fit in the LLM pipeline?

```
Stage 1 — Pretraining          : Model learns language from raw internet text (GPT-4, LLaMA)
Stage 2 — Domain Adaptation    : Non-instruction fine-tuning on domain text (previous notebook)
Stage 3 — Instruction Tuning   : THIS NOTEBOOK — teach model to follow instructions
Stage 4 — Preference Alignment : RLHF / DPO — align model with human preferences
```

## What changes from Non-Instruction Fine-Tuning?

| Aspect | Non-Instruction FT | Instruction FT |
|--------|-------------------|----------------|
| Data format | Raw text | `{instruction, input, output}` pairs |
| What model learns | Domain fluency | How to follow commands |
| Loss computed on | All tokens | **Response tokens only** ← key difference |
| Trainer | `Trainer` | `SFTTrainer` (purpose-built) |
| Prompt template | None needed | Required — must be consistent |

> 💡 **The key insight:** During instruction fine-tuning, we do NOT want the model
> to waste capacity learning to reproduce the *question*. It should only learn to produce
> the *answer*. This is why we mask instruction tokens in the loss.

---
# PART 1 — Install & Imports

In [ ]:
# trl   → SFTTrainer, built specifically for instruction fine-tuning
# peft  → LoRA adapters
# bitsandbytes → 4-bit quantization (QLoRA)
# accelerate   → multi-device abstraction

!pip install -q -U transformers peft bitsandbytes accelerate trl datasets

In [ ]:
import torch
import gc
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)
from trl import SFTTrainer, SFTConfig

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
# PART 2 — Load the Base Model

We build on top of the domain-adapted model from the previous notebook (Non-Instruction FT).  
This is the correct real-world pipeline:

```
TinyLlama (pretrained) → + domain knowledge (NIF) → + instruction following (this notebook)
```

If you don't have the NIF checkpoint, you can also start directly from the base `TinyLlama` — instruction tuning still works, it just won't have domain knowledge baked in.

In [ ]:
# Base model name — same as the previous notebook
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# Path to your NIF checkpoint (from previous notebook)
# If you don't have it, set nif_checkpoint_path = None
# and we'll use the raw base model instead
nif_checkpoint_path = "/content/qlora-finetuned/final-adapter"  # from NIF notebook

print(f"Base model : {model_name}")
print(f"NIF adapter: {nif_checkpoint_path}")

In [ ]:
# ── Load Tokenizer ────────────────────────────────────────────────────────────

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# IMPORTANT: padding_side="right" is required for instruction tuning
# "left" padding is used for generation (inference), but "right" is
# required during training so the loss mask aligns correctly.
tokenizer.padding_side = "right"

print(f"Vocab size    : {tokenizer.vocab_size}")
print(f"Pad token     : {tokenizer.pad_token!r}")
print(f"Padding side  : {tokenizer.padding_side}")

---
# PART 3 — Prepare the Instruction Dataset

Instruction data comes in `{instruction, input, output}` format.  
We need to format it into a single text string using a **prompt template**.

> ⚠️ **Critical rule:** The template you use for training MUST be the exact same  
> template you use at inference. If they differ, the model will perform poorly  
> because it was never trained to respond to that format.

In [ ]:
# ── Option A: Public Instruction Dataset from HuggingFace ─────────────────────
# Great for learning before you have your own data

# Mental health counseling (Context/Response format)
# dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")

# Classic instruction format (instruction/input/output)
# dataset = load_dataset("yahma/alpaca-cleaned", split="train")

# General instruction following
# dataset = load_dataset("tatsu-lab/alpaca", split="train")

# ── Option B: Your Own CSV ─────────────────────────────────────────────────────
# Expected columns: instruction, input, output
# dataset = load_dataset("csv", data_files="/content/pharma_instruction_data.csv", split="train")

# ── For this notebook we use alpaca-cleaned as a concrete example ──────────────
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
print(dataset)
print("\nSample:")
print(dataset[0])

## Prompt Templates — The Right and Wrong Way

There are several common templates. The key is to **pick one and be consistent**.  
We will see what happens when you break consistency.

In [ ]:
# ── Attempt 1: Inconsistent templates — WRONG ─────────────────────────────────
#
# ❌ PROBLEM: Training uses [INST]...[/INST] format,
#             but inference uses a bare prompt with no template.
#
# The model learns: "when I see [INST], I should answer after [/INST]"
# At inference, there is no [INST] → model is confused → bad output.
#
# This is the exact mistake in the original notebook.

def format_train_wrong(example):
    # Training format uses special tokens
    example["text"] = f"[INST] {example['instruction']} [/INST] {example['output']}"
    return example

# At inference (original notebook):
inference_prompt_wrong = "List two advantages of combining Atorvastatin with Ezetimibe."
# ↑ No [INST] token! Model never saw this format during training.

print("Training format:")
print(format_train_wrong(dict(dataset[0]))["text"][:200])
print()
print("Inference format (wrong — no template):")
print(inference_prompt_wrong)
print()
print("❌ These two formats do NOT match — model will underperform at inference.")

In [ ]:
# ── Attempt 2: Consistent template — CORRECT ──────────────────────────────────
#
# ✅ FIX: Use the same template at both training and inference.
#
# We use the Alpaca-style template — clean, widely used, easy to parse.
# The ### markers are easy for the model to learn to look for.
#
# TEMPLATE CHOICES (pick one and stick to it):
#
#   Alpaca style (this notebook):
#     ### Instruction:\n{instruction}\n### Input:\n{input}\n### Response:\n{output}
#
#   Llama-2 chat style:
#     <s>[INST] {instruction}\n{input} [/INST] {output} </s>
#
#   ChatML style (used by Mistral, Qwen):
#     <|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>

PROMPT_TEMPLATE = (
    "### Instruction:\n{instruction}\n"
    "### Input:\n{input}\n"
    "### Response:\n{output}"
)

# For inference (no output — model must generate it)
INFERENCE_TEMPLATE = (
    "### Instruction:\n{instruction}\n"
    "### Input:\n{input}\n"
    "### Response:\n"   # ← no output, model completes from here
)

def format_example(example):
    """Format one row into the prompt template."""
    text = PROMPT_TEMPLATE.format(
        instruction=example.get("instruction", ""),
        input=example.get("input", ""),
        output=example.get("output", "")
    )
    return {"text": text}

formatted = dataset.map(format_example, remove_columns=dataset.column_names)

print("✅ Formatted sample:")
print(formatted[0]["text"])
print()
print(f"Total samples: {len(formatted)}")

In [ ]:
# ── Train / Eval Split ────────────────────────────────────────────────────────
#
# ❌ Original notebook: no validation set at all.
#    Without it, you have no way to know if the model is overfitting
#    or if training is actually improving response quality.
#
# ✅ Always split before tokenizing.

split = formatted.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train samples: {len(train_dataset)}")
print(f"Eval  samples: {len(eval_dataset)}")

---
# PART 4 — Tokenization: The Instruction Masking Problem

This is **the most important concept** that separates instruction fine-tuning from non-instruction fine-tuning.

### Why do we need to mask instruction tokens?

During training, loss is computed on every token unless we explicitly tell PyTorch to ignore some.

```
Input:  ### Instruction:\nWhat is aspirin?\n### Response:\nAspirin is a painkiller.
        |___________________________|           |_________________________________|
              INSTRUCTION PART                          RESPONSE PART
              set labels = -100                    set labels = token IDs
              (ignored in loss)                    (trained on)
```

If you train on the instruction part too, the model wastes capacity learning to reproduce questions it was given — it should focus entirely on learning to produce good answers.

In [ ]:
# ── Attempt 1: No masking — WRONG ─────────────────────────────────────────────
#
# ❌ PROBLEM: labels = input_ids.copy()
#    This makes the model train on ALL tokens:
#    - The instruction/input tokens   ← wasteful, we give this during inference anyway
#    - The response tokens            ← this is what we actually want
#    - The padding tokens             ← pure noise
#
# Result: slower convergence, model memorises the prompt format instead of
#         learning to generate quality responses.

def tokenize_naive(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()  # ❌ trains on everything
    return tokens

# Show the problem — inspect label distribution
sample = tokenize_naive(train_dataset[0])
total       = len(sample["labels"])
pad_count   = sample["labels"].count(tokenizer.pad_token_id)
real_count  = total - pad_count

print("Naive tokenization (WRONG):")
print(f"  Total tokens : {total}")
print(f"  Real tokens  : {real_count}  ← ALL trained on (instruction + response mixed)")
print(f"  Pad tokens   : {pad_count}   ← also trained on (pure noise)")
print()
print("❌ Model is penalized for not predicting padding and instruction tokens — meaningless signal.")

In [ ]:
# ── Attempt 2: Mask padding only — BETTER but still incomplete ────────────────
#
# ✅ Better than Attempt 1 — padding is now ignored.
# ❌ Still wrong — the instruction tokens are still trained on.
#    This is what the NIF notebook does, and it's fine for raw text,
#    but NOT for instruction data.

def tokenize_mask_padding_only(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    labels = [
        t if t != tokenizer.pad_token_id else -100
        for t in tokens["input_ids"]
    ]
    tokens["labels"] = labels
    return tokens

sample = tokenize_mask_padding_only(train_dataset[0])
masked   = sample["labels"].count(-100)
trained  = len(sample["labels"]) - masked

print("Padding-masked tokenization (BETTER but not ideal for instruction tuning):")
print(f"  Trained on : {trained} tokens  ← includes both instruction AND response")
print(f"  Masked out : {masked} tokens   ← only padding")
print()
print("⚠️  For instruction fine-tuning, we should ALSO mask the instruction tokens.")

In [ ]:
# ── Attempt 3: Mask padding AND instruction — CORRECT ─────────────────────────
#
# ✅ The correct approach for instruction fine-tuning:
#    1. Find where the response starts (after "### Response:\n")
#    2. Set labels = -100 for everything BEFORE the response
#    3. Set labels = token_id for the response tokens
#    4. Set labels = -100 for padding
#
# The model ONLY learns to generate the response part.

# First, let's find what token IDs "### Response:\n" maps to
RESPONSE_KEY = "### Response:\n"
response_key_ids = tokenizer.encode(RESPONSE_KEY, add_special_tokens=False)
print(f"Response key tokens: {response_key_ids}")
print(f"Decoded back: {tokenizer.decode(response_key_ids)!r}")
print()

def tokenize_with_response_mask(example):
    """
    Tokenize and mask everything except the response.
    Labels = -100 for instruction part + padding.
    Labels = token_id for response part only.
    """
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    input_ids = tokens["input_ids"]

    # Start with all labels masked
    labels = [-100] * len(input_ids)

    # Find the start of the response section
    key_len = len(response_key_ids)
    response_start = None

    for i in range(len(input_ids) - key_len + 1):
        if input_ids[i : i + key_len] == response_key_ids:
            response_start = i + key_len  # first token AFTER the key
            break

    if response_start is not None:
        # Unmask only the response tokens (not padding)
        for j in range(response_start, len(input_ids)):
            if input_ids[j] != tokenizer.pad_token_id:
                labels[j] = input_ids[j]

    tokens["labels"] = labels
    return tokens


# Verify the masking is working correctly
sample_text = train_dataset[0]["text"]
sample_tok  = tokenize_with_response_mask(train_dataset[0])

total    = len(sample_tok["labels"])
masked   = sample_tok["labels"].count(-100)
trained  = total - masked

print("✅ Response-masked tokenization (CORRECT):")
print(f"  Total tokens         : {total}")
print(f"  Trained on           : {trained}  ← response tokens only")
print(f"  Masked (-100)        : {masked}  ← instruction + padding")
print()

# Decode just the trained tokens so we can verify
trained_tokens = [
    t for t in sample_tok["labels"] if t != -100
]
print("Trained tokens decode to:")
print(tokenizer.decode(trained_tokens))
print()
print("✅ Only the response part — exactly what we want.")

In [ ]:
# ── Apply Correct Tokenization to Full Dataset ────────────────────────────────
#
# ✅ remove_columns must list ALL original string columns
#    The original notebook missed this, leaving stale columns in the dataset
#    which caused shape errors during Trainer collation.

tokenized_train = train_dataset.map(
    tokenize_with_response_mask,
    batched=False,
    remove_columns=train_dataset.column_names   # ✅ remove "text" and any other string cols
)
tokenized_eval = eval_dataset.map(
    tokenize_with_response_mask,
    batched=False,
    remove_columns=eval_dataset.column_names
)

print(tokenized_train)
print("\nColumns:", tokenized_train.column_names)
print("\n✅ Only model-ready tensor columns remain.")

---
# PART 5 — Load Model with QLoRA

Same QLoRA setup as the NIF notebook, but we fix the **order bug** that was
repeated in the original instruction tuning code.

**Correct order (always):**
```
1. Create BitsAndBytesConfig
2. Load model WITH quantization_config
3. Call prepare_model_for_kbit_training
4. Apply LoRA with get_peft_model
```

In [ ]:
# ── Step 1: Load the NIF-adapted base model ───────────────────────────────────
#
# We load the domain-adapted model from the previous notebook as our starting point.
# This gives us a model that already knows pharmaceutical domain language,
# and we will now teach it to FOLLOW INSTRUCTIONS on top of that knowledge.
#
# If you don't have the NIF checkpoint, skip this and use model_name directly.

import os

if nif_checkpoint_path and os.path.exists(nif_checkpoint_path):
    print("Loading NIF-adapted model as base...")
    # Load base model first, then load NIF LoRA adapter on top
    _base = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="cpu"        # load to CPU first before quantizing
    )
    domain_model = PeftModel.from_pretrained(_base, nif_checkpoint_path)
    # Merge the NIF LoRA weights into the base model permanently
    # This creates a single clean model we can then quantize
    domain_model = domain_model.merge_and_unload()
    print("✅ NIF adapter merged into base model.")
    del _base
    gc.collect()
    # Save merged model to a temp path so we can reload it quantized
    merged_path = "/content/domain-merged-model"
    domain_model.save_pretrained(merged_path)
    tokenizer.save_pretrained(merged_path)
    del domain_model
    gc.collect()
    torch.cuda.empty_cache()
    base_model_path = merged_path
else:
    print("NIF checkpoint not found. Using raw base model.")
    base_model_path = model_name

print(f"\nBase model path for instruction training: {base_model_path}")

In [ ]:
# ── Step 2: Create BitsAndBytesConfig FIRST ───────────────────────────────────
#
# ❌ Original notebook mistake (repeated from NIF notebook):
#    prepare_model_for_kbit_training was called on non_instruction_model
#    (which was loaded in full precision, not quantized),
#    THEN a new quantized model was loaded — overwriting the prepared model!
#
# ✅ Correct: create bnb_config FIRST, load model WITH it,
#    then prepare, then apply LoRA.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("✅ BitsAndBytesConfig created. Now loading quantized model...")

In [ ]:
# ── Step 3: Load Quantized Model ──────────────────────────────────────────────

model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    quantization_config=bnb_config,
    device_map={"":0}       # all layers → GPU 0 (training-safe, avoids device mismatch)
)

model.config.use_cache = False          # required for gradient checkpointing
model.enable_input_require_grads()      # required for LoRA on quantized models

mem = torch.cuda.memory_allocated() / 1e9
print(f"✅ Quantized model loaded. GPU memory: {mem:.2f} GB")

In [ ]:
# ── Step 4: Prepare for kbit Training ────────────────────────────────────────
#
# This must be called on the QUANTIZED model (step 3).
# It casts layer norms to fp32 for numerical stability.

model = prepare_model_for_kbit_training(model)
print("✅ Model prepared for kbit training.")

In [ ]:
# ── Step 5: Apply LoRA ────────────────────────────────────────────────────────
#
# ❌ Original notebook only used ["q_proj", "v_proj"]
#    This is the minimal set from the original LoRA paper.
#
# ✅ Adding k_proj and o_proj gives better gradient coverage
#    and faster convergence, especially on small datasets.

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                                                        # rank
    lora_alpha=16,                                              # scale = alpha/r = 2.0
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],   # ✅ all 4 attention projections
    lora_dropout=0.05,
    bias="none"
)

qlora_model = get_peft_model(model, lora_config)
qlora_model.print_trainable_parameters()
# Expected: ~0.5-1% of total parameters — this is the point of LoRA

---
# PART 6 — Baseline Inference (Before Training)

**Always run inference BEFORE training.**  
Without a baseline, you cannot measure what fine-tuning actually changed.

This was completely missing from the original notebook.

In [ ]:
# ── Baseline: What does the model say BEFORE instruction tuning? ──────────────

def generate(model, instruction: str, input_text: str = "", max_new_tokens: int = 150) -> str:
    """
    Generate using the SAME template as training.
    This is the correct way — same format at train and inference time.
    """
    prompt = INFERENCE_TEMPLATE.format(
        instruction=instruction,
        input=input_text
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    # Decode only the NEW tokens (not the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


test_cases = [
    {
        "instruction": "List two advantages of combining Atorvastatin with Ezetimibe.",
        "input": ""
    },
    {
        "instruction": "Explain the mechanism of action of Metformin.",
        "input": ""
    },
    {
        "instruction": "What are common side effects of beta-blockers?",
        "input": "Patient is 65 years old with hypertension."
    },
]

qlora_model.eval()
baseline_outputs = {}

for i, tc in enumerate(test_cases):
    output = generate(qlora_model, tc["instruction"], tc["input"])
    baseline_outputs[i] = output
    print(f"{'='*60}")
    print(f"TEST {i+1}: {tc['instruction']}")
    print(f"{'-'*60}")
    print(f"BASELINE OUTPUT:")
    print(output)
    print()

print("📝 Baseline outputs saved. We will compare these after training.")

---
# PART 7 — Training

## Raw `Trainer` vs `SFTTrainer` — Why does it matter?

| Feature | Raw `Trainer` | `SFTTrainer` |
|---------|--------------|-------------|
| Response masking | Manual (error-prone) | Built-in via `dataset_text_field` |
| Sequence packing | Not supported | Supported (`packing=True`) |
| LoRA integration | Manual `get_peft_model` | Pass `peft_config` directly |
| Instruction formats | Manual formatting | Supports ChatML natively |

We show BOTH approaches below — raw `Trainer` first to understand what SFTTrainer does for you,  
then `SFTTrainer` as the correct solution.

In [ ]:
# ── Attempt 1: Raw Trainer — possible but fragile ─────────────────────────────
#
# Using the raw Trainer for instruction data is not wrong per se,
# but it requires you to manually handle:
#   - Response masking (which we did in Part 4)
#   - Consistent prompt formatting
#   - Sequence packing (efficiency)
#
# ❌ The original notebook used raw Trainer WITH:
#    - No response masking
#    - No eval dataset
#    - Wrong LR (2e-5 instead of 2e-4)
#    - No warmup
#    - No gradient accumulation
#
# We define the correct raw Trainer args here for reference,
# but we will USE SFTTrainer below.

from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

raw_training_args = TrainingArguments(
    output_dir="./instruction-raw-trainer",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,      # effective batch = 16
    learning_rate=2e-4,                 # ✅ higher LR for LoRA (was 2e-5 in original)
    warmup_steps=50,                    # ✅ prevents early instability
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    evaluation_strategy="steps",        # ✅ monitor eval loss
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# raw_trainer = Trainer(
#     model=qlora_model,
#     args=raw_training_args,
#     train_dataset=tokenized_train,   # ← uses our manually masked dataset
#     eval_dataset=tokenized_eval,
#     data_collator=data_collator
# )
# raw_trainer.train()  # This would work correctly but SFTTrainer is cleaner

print("Raw Trainer defined for reference.")
print("Proceeding to SFTTrainer — the purpose-built solution.")

In [ ]:
# ── Attempt 2: SFTTrainer — the correct tool for instruction fine-tuning ───────
#
# ✅ SFTTrainer (Supervised Fine-Tuning Trainer) from trl:
#
#   1. Takes raw formatted text via dataset_text_field="text"
#      → handles tokenization internally
#
#   2. Takes peft_config directly
#      → handles prepare_model_for_kbit_training + get_peft_model internally
#      (so you can skip Part 5 if using SFTTrainer from scratch)
#
#   3. max_seq_length truncates sequences correctly
#
#   4. packing=True (optional) packs multiple short samples into one sequence
#      for GPU efficiency — great for datasets with short instructions
#
# NOTE: When using SFTTrainer's built-in tokenization (dataset_text_field),
# you pass the RAW formatted dataset (not the tokenized_train from Part 4).
# SFTTrainer does the tokenization itself.
# If you already tokenized manually (Part 4), pass tokenized_train and
# do NOT set dataset_text_field.

sft_config = SFTConfig(
    output_dir="./instruction-sft",
    dataset_text_field="text",          # SFTTrainer reads from this column
    max_seq_length=512,
    packing=False,                      # True = pack short sequences (faster, ~10-20% more efficient)
                                        # False = one sample per sequence (simpler to debug)

    # ── Training schedule ───────────────────────────────────────────────
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,      # effective batch = 16

    # ── Optimizer ───────────────────────────────────────────────────────
    learning_rate=2e-4,                 # ✅ correct LR for LoRA
    warmup_steps=50,
    optim="paged_adamw_8bit",

    # ── Memory ──────────────────────────────────────────────────────────
    fp16=True,
    gradient_checkpointing=True,

    # ── Evaluation & saving ─────────────────────────────────────────────
    evaluation_strategy="steps",
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,

    # ── Logging ─────────────────────────────────────────────────────────
    logging_steps=50,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=qlora_model,
    args=sft_config,
    train_dataset=train_dataset,        # raw formatted dataset (not tokenized)
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    peft_config=lora_config,
)

print("✅ SFTTrainer ready.")

In [ ]:
# ── Start Training ────────────────────────────────────────────────────────────
#
# Watch the logs:
#   train/loss → should decrease
#   eval/loss  → should also decrease, or stay flat
#
# ⚠️ WARNING SIGNS:
#   train/loss going down + eval/loss going UP = OVERFITTING
#   Solution: reduce epochs, add more data, increase lora_dropout
#
#   Both losses flat = LR too low or data too similar to pretraining data
#   Solution: increase learning_rate, check data quality

qlora_model.train()
sft_trainer.train()

print("\n✅ Training complete!")

In [ ]:
# ── Save the Instruction-Tuned LoRA Adapter ───────────────────────────────────
#
# We save ONLY the adapter (~10-50MB), not the full model (~2GB).
# To use it later: load base model + load this adapter on top.

adapter_path = "./instruction-sft/final-adapter"
qlora_model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"✅ Adapter saved to: {adapter_path}")

# Zip and download
import zipfile, os

zip_path = "/content/instruction_adapter.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(adapter_path):
        for file in files:
            full = os.path.join(root, file)
            zf.write(full, os.path.relpath(full, adapter_path))

print(f"✅ Zipped to: {zip_path}")

---
# PART 8 — Inference: Compare Before vs After

In [ ]:
# ── Load Fine-Tuned Model for Inference ───────────────────────────────────────
#
# ❌ Original notebook mistake:
#    Training used device_map={"":0} but inference used device_map="auto".
#    This can cause tensor device mismatch errors.
#
# ✅ Keep device_map consistent between training and inference.

gc.collect()
torch.cuda.empty_cache()

base_for_infer = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,
    device_map={"":0}           # ✅ consistent with training
)

instruction_model = PeftModel.from_pretrained(base_for_infer, adapter_path)
instruction_model.eval()

print("✅ Instruction-tuned model loaded for inference.")

In [ ]:
# ── Compare Baseline vs Fine-Tuned ────────────────────────────────────────────
#
# Key things to check:
#   1. Does the fine-tuned model follow the instruction more precisely?
#   2. Does it use domain vocabulary correctly?
#   3. Does it stay on topic instead of rambling?
#   4. Does the format match what was trained on?

for i, tc in enumerate(test_cases):
    finetuned_output = generate(instruction_model, tc["instruction"], tc["input"])

    print(f"{'='*65}")
    print(f"TEST {i+1}: {tc['instruction']}")
    if tc["input"]:
        print(f"INPUT  : {tc['input']}")
    print(f"{'─'*65}")
    print("BEFORE (baseline):")
    print(baseline_outputs[i])
    print()
    print("AFTER (instruction fine-tuned):")
    print(finetuned_output)
    print()

---
# PART 9 — Merge Adapter into Base Model (Optional)

After fine-tuning, you can **merge** the LoRA adapter weights permanently into the base model.

| | Separate (adapter + base) | Merged |
|--|--------------------------|--------|
| Inference speed | Slightly slower | Faster |
| Storage | Base + small adapter | One larger file |
| Can undo fine-tuning? | Yes — discard adapter | No |
| HuggingFace Hub | Upload adapter only | Upload full model |

**Merge when:** you are done experimenting and want production deployment.  
**Keep separate when:** you are still iterating or want to swap adapters.

In [ ]:
# ── Merge LoRA Adapter into Base Model ───────────────────────────────────────

print("Merging LoRA adapter into base model...")
merged_model = instruction_model.merge_and_unload()

merged_save_path = "/content/instruction-merged-model"
merged_model.save_pretrained(merged_save_path)
tokenizer.save_pretrained(merged_save_path)

print(f"✅ Merged model saved to: {merged_save_path}")
print("   This model can be loaded directly without PEFT.")
print()
print("To load the merged model later:")
print("  model = AutoModelForCausalLM.from_pretrained(merged_save_path)")
print("  (No PeftModel needed — it's a regular model now)")

In [ ]:
# ── Optional: Push to HuggingFace Hub ────────────────────────────────────────
#
# You can share your fine-tuned model publicly or privately.
# Login with your HF token first: huggingface-cli login
#
# Option A: Push just the adapter (small, ~50MB)
# instruction_model.push_to_hub("your-username/tinyllama-pharma-instruct")
#
# Option B: Push the merged model (full model, ~2GB)
# merged_model.push_to_hub("your-username/tinyllama-pharma-instruct-merged")

print("Uncomment the push_to_hub lines above when you're ready to share.")

---
# PART 10 — Summary: What We Learned

| Step | ❌ Wrong (original notebook) | ✅ Fixed | Why It Matters |
|------|------------------------------|---------|----------------|
| **LoRA target** | Applied on `non_instruction_model` (not quantized) | Apply on freshly loaded quantized model | LoRA on a non-quantized model wastes memory and produces wrong gradients |
| **bnb_config order** | `prepare_kbit` before loading bnb model | `bnb_config → load → prepare_kbit` | Preparing a non-quantized model does nothing useful |
| **Label masking** | `labels = input_ids.copy()` | `-100` for padding AND instruction tokens | Model should only learn to generate responses, not reproduce prompts |
| **Prompt template** | Train: `[INST]...[/INST]`, Infer: bare prompt | Same template at both train and inference | Format mismatch at inference causes bad outputs |
| **Trainer choice** | Raw `Trainer` with manual tokenization | `SFTTrainer` purpose-built for instruction data | SFTTrainer handles packing, masking, and LoRA integration |
| **Learning rate** | `2e-5` (full fine-tuning rate) | `2e-4` (correct for LoRA adapters) | LoRA adapters are freshly initialized and need higher LR |
| **Eval dataset** | None | 95/5 train/eval split | Can't detect overfitting without validation loss |
| **Baseline** | Not done | Run inference before training | Can't measure improvement without a before/after |
| **remove_columns** | Missing — stale columns remain | `remove_columns=dataset.column_names` | Leftover string columns cause shape errors in collation |
| **device_map** | Train `{"":0}` / Infer `auto` (mismatch) | Both `{"":0}` | Tensor device mismatch errors at inference |

---

## The Full 3-Stage Pipeline So Far

```
TinyLlama (pretrained on internet text)
        ↓
[NIF Notebook]   Domain adaptation on pharmaceutical PDFs
        ↓         (model learns domain vocabulary and facts)
[This Notebook]  Instruction fine-tuning on {instruction, output} pairs
        ↓         (model learns to follow commands in the domain)
[Next Notebook]  Preference alignment (DPO/RLHF)
                  (model learns to prefer better responses over worse ones)
```

## Next Steps

1. **Increase dataset size** — more diverse instruction pairs = better generalization
2. **Try `packing=True`** in SFTTrainer for ~15% training speedup
3. **Experiment with `r=16` or `r=32`** if you have more VRAM — higher rank = more capacity
4. **Preference Alignment (next notebook)** — use `DPOTrainer` from `trl` to align responses with human preferences